In [39]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.model_selection import train_test_split

In [40]:
data = pd.read_csv('Churn_Modelling.csv')
data.head()

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15945405,Lewis,567,France,Male,58,0,13674.54,1,1,1,144022.31,0
1,2,15951000,Williams,571,Germany,Male,43,8,0.00,2,1,1,65399.77,0
2,3,15630085,Jones,628,Spain,Female,30,1,0.00,3,1,1,63760.04,0
3,4,15830429,Thomas,685,Spain,Female,44,8,107540.45,1,0,0,55274.68,1
4,5,15615278,King,737,France,Female,18,9,204248.69,2,0,0,176273.19,0


In [41]:
## Preprocess the Data

data = data.drop(['RowNumber','CustomerId','Surname'],axis = 1)

In [42]:
data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,567,France,Male,58,0,13674.54,1,1,1,144022.31,0
1,571,Germany,Male,43,8,0.00,2,1,1,65399.77,0
2,628,Spain,Female,30,1,0.00,3,1,1,63760.04,0
3,685,Spain,Female,44,8,107540.45,1,0,0,55274.68,1
4,737,France,Female,18,9,204248.69,2,0,0,176273.19,0
...,...,...,...,...,...,...,...,...,...,...,...
995,596,France,Female,54,8,215021.93,2,1,1,168623.76,0
996,747,Germany,Female,28,3,0.00,1,1,1,65839.12,0
997,848,France,Male,39,8,0.00,1,1,1,20811.28,0
998,499,Germany,Male,34,1,35779.44,2,1,1,19973.51,1


In [43]:
## label encoder
label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])

In [44]:
## One Hot Enocde 'GeoGraphy'ArithmeticError
ohe_geo = OneHotEncoder(handle_unknown='ignore')
geo_encoded = ohe_geo.fit_transform(data[['Geography']]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded,columns=ohe_geo.get_feature_names_out(['Geography']))


In [45]:
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,1.0,0.0
2,0.0,0.0,1.0
3,0.0,0.0,1.0
4,1.0,0.0,0.0
...,...,...,...
995,1.0,0.0,0.0
996,0.0,1.0,0.0
997,1.0,0.0,0.0
998,0.0,1.0,0.0


In [46]:
## Combine ohe-hot encoded column with original data
data = pd.concat([data.drop('Geography',axis=1),geo_encoded_df],axis=1)
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,567,1,58,0,13674.54,1,1,1,144022.31,0,1.0,0.0,0.0
1,571,1,43,8,0.00,2,1,1,65399.77,0,0.0,1.0,0.0
2,628,0,30,1,0.00,3,1,1,63760.04,0,0.0,0.0,1.0
3,685,0,44,8,107540.45,1,0,0,55274.68,1,0.0,0.0,1.0
4,737,0,18,9,204248.69,2,0,0,176273.19,0,1.0,0.0,0.0


In [47]:
## Split the data into features and target

X = data.drop('EstimatedSalary', axis=1)
y = data['EstimatedSalary']


In [48]:
## Split the data in training and testing sets

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2, random_state=42)

In [49]:
scaler = StandardScaler()
X_train= scaler.fit_transform(X_train)
X_test= scaler.transform(X_test)

In [52]:
with open('label_encoder_gender.pkl','wb') as file:
    pickle.dump(label_encoder_gender,file)

with open('ohe_geo.pkl','wb') as file:
    pickle.dump(ohe_geo,file)

with open('scaler.pkl','wb') as file:
    pickle.dump(scaler,file)


## ANN Regression Problem Statement


In [53]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [54]:
## Build the Model
from sklearn import metrics


model = Sequential([
    Dense(64, activation='relu',input_shape=(X_train.shape[1],)),
    Dense(32, activation='relu'),
    Dense(1) ## Output Layer for Regression
])

## Compile the model
model.compile(optimizer='adam',loss='mean_absolute_error',metrics=['mae'])

model.summary()

f:\PYTHON_AI\.venv\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           832 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,945 (11.50 KB)

 Trainable params: 2,945 (11.50 KB)

 Non-trainable params: 0 (0.00 B)

In [61]:
import tensorboard
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

## Set up TensorBoard
log_dir = "regressionlogs/fit/" +datetime.datetime.now().strftime("%Y%m/%d-%H%M%S")
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

In [62]:
## Setup EarlyStopping
early_stopping_calback =EarlyStopping(monitor='val_loss',patience=10,restore_best_weights=True)

In [63]:
history = model.fit(
    X_train,y_train,
    validation_data=(X_test,y_test),
    epochs=100,
    callbacks=[early_stopping_calback,tensorboard_callback]
    )

Epoch 1/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 3s 64ms/step - loss: 49458.0234 - mae: 49458.0234 - val_loss: 54381.5742 - val_mae: 54381.5742
Epoch 2/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 49299.7812 - mae: 49299.7812 - val_loss: 54133.9609 - val_mae: 54133.9609
Epoch 3/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 49137.8086 - mae: 49137.8086 - val_loss: 53912.8867 - val_mae: 53912.8867
Epoch 4/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step - loss: 48998.5508 - mae: 48998.5508 - val_loss: 53699.1367 - val_mae: 53699.1367
Epoch 5/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - loss: 48859.8789 - mae: 48859.8789 - val_loss: 53501.8633 - val_mae: 53501.8633
Epoch 6/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - loss: 48729.5000 - mae: 48729.5000 - val_loss: 53284.2852 - val_mae: 53284.2852
Epoch 7/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - loss: 48601.7969 - mae: 48601.7969 - val_loss: 53107.0156 - val_mae: 53107.0156
Epoch 8/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 48

In [64]:
## Evaluate the model
%load_ext tensorboard

In [67]:
%tensorboard --logdir regressionlogs/fit

Reusing TensorBoard on port 6006 (pid 20772), started 0:03:05 ago. (Use '!kill 20772' to kill it.)

In [69]:
 ## Evaluate model on test data

test_loss,tes_mae = model.evaluate(X_test,y_test)
print(f'Test Mae: {tes_mae}')

7/7 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - loss: 50219.7344 - mae: 50219.7344
Test Mae: 50219.734375


In [70]:
model.save('regression_model.h5')